In [ ]:
import os
import re
from dotenv import load_dotenv
from langchain_core.documents import Document
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from uuid import uuid4
from langchain_groq import ChatGroq

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser


In [ ]:
load_dotenv()
groq_api = os.getenv('GROQ_API_KEY')

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.7
)

In [ ]:
def load_youtube_snippets(url: str, window_seconds: int = 60, overlap_seconds: int = 10):

    match = re.search(r"(?:v=|youtu\.be/)([\w-]+)", url)

    if not match:
        raise ValueError(f"Could not extract video ID from URL: {url}")
    
    video_id = match.group(1)
    transcript = YouTubeTranscriptApi().fetch(video_id, languages=['en', 'bn', 'hi'])
    snippets = transcript.snippets
    
    chunks = []
    current_text = []
    window_start = snippets[0].start if snippets else 0.0
    last_end = window_start

    for snippet in snippets:
        current_text.append(snippet.text)
        last_end = snippet.start + snippet.duration

        if last_end - window_start >= window_seconds:
            chunks.append(Document(
                page_content=" ".join(current_text).strip(),
                metadata={"source": video_id, "start": round(window_start, 2), "end": round(last_end, 2)}
            ))
            overlap_start = max(window_start, last_end - overlap_seconds)
            current_text = [s.text for s in snippets if overlap_start <= s.start < last_end]
            window_start = overlap_start

    if current_text:
        chunks.append(Document(
            page_content=" ".join(current_text).strip(),
            metadata={"source": video_id, "start": round(window_start, 2), "end": round(last_end, 2)}
        ))

    return chunks

texts = load_youtube_snippets("https://www.youtube.com/watch?v=tL9Lw250spc")
whole_content = ' '.join(doc.page_content for doc in texts)


In [ ]:
pc = Pinecone()
index_name = "rag-extention"  # change if desired

if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

index = pc.Index(index_name)

vector_store = PineconeVectorStore(index=index, embedding=embedding_model)

In [50]:
uuids = [str(uuid4()) for _ in range(len(texts))]
vector_store.add_documents(documents=texts, ids=uuids)

['53df7855-add2-4e43-bbab-481200928f4c',
 'c996822a-7450-41a4-b379-f16463bf8b79',
 'aded704a-f23a-458c-935c-c030ef157ae8',
 '319485b5-1ed4-4660-a7c5-def4fa28aaaa',
 'd0130a7c-a1fb-4118-aca9-4a3a2c836309',
 '612e49bb-bc25-44d3-86ba-272d94a08d43',
 '04f59f0e-ba23-4059-93b3-55bcbdc9eb07',
 '8bda50d2-591a-4f4d-a656-620a2f97e8a7',
 'dfd0f14b-1932-4d03-95de-d1aa60fb7ac4',
 'aadbb145-91d3-41de-8772-cad52e0a792a',
 'aa5348fb-2c14-4af6-9b85-ce45761102d4',
 '6c906171-fa9c-4946-9e0d-cbfd6e29e399',
 '14f17f8e-e6ae-4425-ae7b-d2209d5b4f80',
 '4a861b16-4d55-4c49-9254-734b05642e41',
 '4bb5707d-36e0-46cb-acc9-1aab0349246c',
 '012c14f1-adbb-4339-b203-5bb1a14ee082',
 '3ce3354e-1940-46f5-b13a-1039e86864c4',
 '55413d79-1172-4497-a8b5-bb7f4a8fefff',
 '0ddad783-40ef-4a71-92c1-2d5c45b18352',
 '442c9a81-5dd4-4277-8fac-a99d10f06de3',
 'db72670b-ba65-4d47-b43a-423f01b208e8',
 '83a212cc-ab4b-47af-93d0-6a14d4c89f22',
 '4107b61c-4f20-4f87-b75b-435334153842',
 '493cbf68-0b4c-40ad-99e4-5744578854dd',
 '13e84757-593b-

In [ ]:
parser = StrOutputParser()

prompt = PromptTemplate(
    template="""## You act as an YouTube Video Summarizer.\n
    Answer the query form the given description.\n query: {query},\n description: {description}
""",
input_variables=['query', 'description']
)

In [80]:
query =  "What is this video about"

In [81]:
results = vector_store.similarity_search_with_score( query, k=3 )

fetch_data = "\n\n".join(res.page_content for res, _ in results)

chain = prompt | llm | parser

output = chain.invoke({
    'query': query,
    'description': fetch_data
})

In [82]:
print(output)

The video appears to be about the story behind the discovery of a mathematical explanation for Kleiber's Law, which relates to biological scaling relationships. Specifically, it discusses how a team of researchers, including Jim, Geoffrey, West, Brown, and Enquist, came together to develop a formal mathematical framework to explain the law, which describes how the metabolic rate of organisms scales with their size. The video seems to be an educational science video, likely from the Veritasium channel, and is sponsored by Brilliant, an online learning platform.


In [ ]:
"""
The video appears to be about the story behind the discovery of a mathematical explanation for Kleiber's 
Law, which relates to biological scaling relationships. Specifically, it discusses how a team of researchers, 
including Jim, Geoffrey, West, Brown, and Enquist, came together to develop a formal mathematical framework to 
explain the law, which describes how the metabolic rate of organisms scales with their size. The video seems to 
be an educational science video, likely from the Veritasium channel, and is sponsored by Brilliant, an online 
learning platform.
"""